In [ ]:
# Check GPU availability
!nvidia-smi

# Install required packages
!pip install transformers torch tqdm -q

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Sun Mar 15 20:41:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
DATA_DIR = '/content/'
OUTPUT_DIR = '/content/embeddings'

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
import logging

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Model configuration
MODEL_NAME = "multilingual-e5-large-intertext"
MODEL_ID = "julian-schelb/multilingual-e5-large-emb-lat-intertext-v1"
BATCH_SIZE = 32
MAX_LENGTH = 512  # Maximum sequence length

print(f"Model: {MODEL_NAME}")
print(f"Model ID: {MODEL_ID}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LENGTH}")

Model: multilingual-e5-large-intertext
Model ID: julian-schelb/multilingual-e5-large-emb-lat-intertext-v1
Batch size: 32
Max length: 512


In [ ]:
def load_chunks_json(filepath: str) -> Tuple[List[Dict], Dict]:
    """
    Load chunks from JSON file.

    Args:
        filepath: Path to the .json file (with or without extension)

    Returns:
        Tuple of (chunks list, metadata dict)
    """
    # Ensure .json extension
    if not filepath.endswith('.json'):
        filepath = f"{filepath}.json"

    print(f"Loading {Path(filepath).name}...")

    # Load JSON file
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract chunks and metadata
    chunks = data.get('chunks', [])
    metadata = data.get('metadata', {})

    print(f"✓ Loaded {len(chunks):,} chunks")

    # Show retention rate if available in metadata
    if metadata.get('filtering', {}).get('retention_rate'):
        retention = metadata['filtering']['retention_rate'] * 100
        print(f"  Retention rate: {retention:.1f}%")

    print()

    return chunks, metadata


# Load BDC and PSC chunks
#bdc_chunks, bdc_metadata = load_chunks_json(f"{DATA_DIR}/VD_bdc_chunks.json")
psc_chunks, psc_metadata = load_chunks_json(f"{DATA_DIR}/VD_psc_chunks.json")

print(f"Total chunks to embed:")
#print(f"  BDC (queries): {len(bdc_chunks):,}")
print(f"  PSC (candidates): {len(psc_chunks):,}")

Loading VD_psc_chunks.json...
✓ Loaded 46,408 chunks

Total chunks to embed:
  PSC (candidates): 46,408


In [ ]:
print(f"Loading {MODEL_NAME}.")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()  # ??

print("Model loaded successfully")
print(f"  Model on device: {next(model.parameters()).device}")

Loading multilingual-e5-large-intertext.


config.json:   0%|          | 0.00/742 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully
  Model on device: cuda:0


In [ ]:
def encode_batch(texts: List[str]) -> np.ndarray:
    """
    Args:
        texts: List of text strings

    Returns:
        Numpy array of embeddings [batch_size, 768]
    """
    # Tokenize
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(device)

    # Generate embeddings
    with torch.no_grad():
        outputs = model(**encoded)

        # SPhilBERTa, use pooler output or [CLS] token
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            embeddings = outputs.pooler_output
        else:
            # Fallback: Use [CLS] token (first token)
            embeddings = outputs.last_hidden_state[:, 0, :]

    # Normalize embeddings for cosine similarity
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    return embeddings.cpu().numpy()


def encode_corpus(
    chunks: List[Dict],
    corpus_name: str,
    checkpoint_every: int = 1000
) -> np.ndarray:
    """
    Encode entire corpus in batches with checkpointing.

    Args:
        chunks: List of chunk dictionaries
        corpus_name: Name for logging (BDC/PSC)
        checkpoint_every: Save checkpoint every N batches

    Returns:
        Numpy array of embeddings [num_chunks, 768]
    """
    print(f"\n{'='*60}")
    print(f"Encoding {corpus_name}")
    print(f"{'='*60}")
    print(f"Total chunks: {len(chunks):,}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Estimated batches: {(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}")
    print()

    # Extract texts
    texts = [chunk["text"] for chunk in chunks]

    # Process in batches
    all_embeddings = []

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"Encoding {corpus_name}"):
        batch_texts = texts[i:i + BATCH_SIZE]
        batch_embeddings = encode_batch(batch_texts)
        all_embeddings.append(batch_embeddings)

        # Checkpoint to prevent data loss
        batch_num = i // BATCH_SIZE
        if batch_num > 0 and batch_num % checkpoint_every == 0:
            checkpoint = np.vstack(all_embeddings)
            checkpoint_path = f"{OUTPUT_DIR}/{corpus_name.lower()}_checkpoint.npy"
            np.save(checkpoint_path, checkpoint)
            print(f"\n✓ Checkpoint saved: {checkpoint.shape[0]} embeddings")

    # Stack all embeddings
    embeddings = np.vstack(all_embeddings)

    print(f"\n✓ Encoding complete")
    print(f"  Shape: {embeddings.shape}")
    print(f"  Embedding dimension: {embeddings.shape[1]}")
    print(f"  Memory: {embeddings.nbytes / 1024**2:.2f} MB")

    return embeddings

In [ ]:
bdc_embeddings = encode_corpus(bdc_chunks, "BDC")

# Save BDC embeddings
bdc_output_path = f"{OUTPUT_DIR}/bdc_embeddings_e5.npy"
np.save(bdc_output_path, bdc_embeddings)
print(f"\n✓ Saved BDC embeddings to: {bdc_output_path}")

# Save BDC chunk IDs (for mapping embeddings back to chunks)
bdc_ids = [chunk["chunk_id"] for chunk in bdc_chunks]
bdc_ids_path = f"{OUTPUT_DIR}/bdc_ids_m5.json"
with open(bdc_ids_path, 'w') as f:
    json.dump(bdc_ids, f, indent=2)
print(f"✓ Saved BDC IDs to: {bdc_ids_path}")


Encoding BDC
Total chunks: 19,466
Batch size: 32
Estimated batches: 609



Encoding BDC:   0%|          | 0/609 [00:00<?, ?it/s]


✓ Encoding complete
  Shape: (19466, 1024)
  Embedding dimension: 1024
  Memory: 76.04 MB

✓ Saved BDC embeddings to: /content/embeddings/bdc_embeddings_e5.npy
✓ Saved BDC IDs to: /content/embeddings/bdc_ids_m5.json


In [ ]:
psc_embeddings = encode_corpus(psc_chunks, "PSC")

# Save PSC embeddings
psc_output_path = f"{OUTPUT_DIR}/psc_embeddings_e5.npy"
np.save(psc_output_path, psc_embeddings)
print(f"\n✓ Saved PSC embeddings to: {psc_output_path}")

# Save PSC chunk IDs
psc_ids = [chunk["chunk_id"] for chunk in psc_chunks]
psc_ids_path = f"{OUTPUT_DIR}/psc_ids_e5.json"
with open(psc_ids_path, 'w') as f:
    json.dump(psc_ids, f, indent=2)
print(f"✓ Saved PSC IDs to: {psc_ids_path}")


Encoding PSC
Total chunks: 46,408
Batch size: 32
Estimated batches: 1451



Encoding PSC:   0%|          | 0/1451 [00:00<?, ?it/s]


✓ Checkpoint saved: 32032 embeddings

✓ Encoding complete
  Shape: (46408, 1024)
  Embedding dimension: 1024
  Memory: 181.28 MB

✓ Saved PSC embeddings to: /content/embeddings/psc_embeddings_e5.npy
✓ Saved PSC IDs to: /content/embeddings/psc_ids_e5.json


In [ ]:
metadata = {
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "device": str(device),
    "bdc": {
        "num_chunks": len(bdc_chunks),
        "embedding_shape": list(bdc_embeddings.shape),
        "embedding_dim": bdc_embeddings.shape[1],
        "file": "bdc_embeddings_e5.npy"
    },
    "psc": {
        "num_chunks": len(psc_chunks),
        "embedding_shape": list(psc_embeddings.shape),
        "embedding_dim": psc_embeddings.shape[1],
        "file": "psc_embeddings_e5-intertext.npy"
    }
}

metadata_path = f"{OUTPUT_DIR}/metadata_e5.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved metadata to: {metadata_path}")

✓ Saved metadata to: /content/embeddings/metadata_e5.json


In [ ]:
#  sanity check
print(f"  BDC embeddings shape matches chunks: {bdc_embeddings.shape[0] == len(bdc_chunks)}")
print(f"  PSC embeddings shape matches chunks: {psc_embeddings.shape[0] == len(psc_chunks)}")
print(f"  BDC IDs count matches: {len(bdc_ids) == len(bdc_chunks)}")
print(f"  PSC IDs count matches: {len(psc_ids) == len(psc_chunks)}")
print(f"  Embedding dimensions match: {bdc_embeddings.shape[1] == psc_embeddings.shape[1]}")

  BDC embeddings shape matches chunks: True
  PSC embeddings shape matches chunks: True
  BDC IDs count matches: True
  PSC IDs count matches: True
  Embedding dimensions match: True
